# Week 5 — Portfolio optimization in Python
## From machine-learning forecasts to constrained decisions

**Guiding question:** *Can a simple machine-learning model improve a constrained portfolio compared with equal weights and historical averages?*

Imagine that you are building a small monthly robo-advisor. At each month end it distributes capital across four asset classes. It first predicts next month's returns and then chooses portfolio weights.


### Learning goals

By the end of this notebook, you will be able to:

- distinguish a return forecast from a portfolio decision;
- align current information with next month's target without looking into the future;
- explain why Ridge regression shrinks coefficients and how `alpha` affects that shrinkage;
- compare a prediction model with a simple baseline using time-ordered validation;
- calculate portfolio return and variance from asset weights;
- formulate bounds and constraints for `scipy.optimize.minimize`;
- compare equal-weight, historical-mean and ML-informed portfolios on future months; and
- explain why precise optimized weights can still be uncertain.

Keep asking: **What is our question? What is our baseline? How do we evaluate the result? What can we conclude?**


<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th colspan="2">Contents</th></tr></thead>
<tbody>
<tr><td>1</td><td><a href="#section-1-the-investment-decision">The investment decision</a></td></tr>
<tr><td>2</td><td><a href="#section-2-meet-the-data">Meet the data</a></td></tr>
<tr><td>3</td><td><a href="#section-3-predict-next-months-returns">Predict next month's returns</a></td></tr>
<tr><td>4</td><td><a href="#section-4-from-forecasts-to-a-portfolio">From forecasts to a portfolio</a></td></tr>
<tr><td>5</td><td><a href="#section-5-optimize-one-portfolio">Optimize one portfolio</a></td></tr>
<tr><td>6</td><td><a href="#section-6-compare-strategies-on-future-months">Compare strategies on future months</a></td></tr>
<tr><td>7</td><td><a href="#section-7-sensitivity-and-limitations">Sensitivity and limitations</a></td></tr>
<tr><td>8</td><td><a href="#section-8-recap-and-outlook">Recap and outlook</a></td></tr>
<tr><td>★</td><td><a href="#section-optional-your-turn">Optional: Your turn</a></td></tr>
</tbody></table>


### How to work with this notebook

Run the cells from top to bottom with **Shift + Enter**. Later cells reuse objects created earlier. At each **Self-check**, stop before reading on and explain your expectation in words.

This notebook contains one connected example. The aim is not to find a universally best investment strategy. It is to understand how a prediction model, assumptions about risk and an optimizer interact.


<h2 id="section-1-the-investment-decision">1. The investment decision</h2>

The robo-advisor allocates capital across four synthetic asset classes:

<table style="margin-left: auto; margin-right: auto;">
  <thead>
    <tr>
      <th>Asset class</th>
      <th>Short description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Global equity</td>
      <td>Higher expected return and higher variation</td>
    </tr>
    <tr>
      <td>Government bonds</td>
      <td>Lower variation and some defensive behaviour</td>
    </tr>
    <tr>
      <td>Corporate bonds</td>
      <td>Between equities and government bonds</td>
    </tr>
    <tr>
      <td>Gold</td>
      <td>A separate source of risk and potential diversification</td>
    </tr>
  </tbody>
</table>

At the end of a month, the advisor knows that month's returns and a market-stress indicator. It predicts the four returns for the **next** month. A constrained optimizer then turns those forecasts into weights.

We compare three strategies:

1. **Equal weight:** always invest 25% in each asset.
2. **Historical optimizer:** optimize using average past returns.
3. **ML optimizer:** optimize using Ridge predictions.

All optimized portfolios must invest all capital, must not use short selling (negative allocations), may invest at most 50% in one asset, and may not exceed the estimated risk of the equal-weight portfolio.

**Self-check:** Which part of this process learns from examples, and which part applies decision rules?  
The Ridge model learns a prediction rule from past months. The optimizer does not learn that rule; it searches for weights that satisfy the stated objective and constraints.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

<h2 id="section-2-meet-the-data">2. Meet the data</h2>

Each row describes one month. Returns are stored as decimals: `0.02` means a 2% gain and `-0.02` means a 2% loss. The `stress_index` lies between 0 and 1; a larger value represents a more stressed synthetic market environment.

`parse_dates` converts the date strings into pandas dates. `set_index` uses those dates as row labels, making the time order visible.


In [ ]:
data_path = Path("data/portfolio_monthly.csv")
data = pd.read_csv(data_path, parse_dates=["date"]).set_index("date")

assets = ["global_equity", "government_bonds", "corporate_bonds", "gold"]
asset_labels = {
    "global_equity": "Global equity",
    "government_bonds": "Government bonds",
    "corporate_bonds": "Corporate bonds",
    "gold": "Gold",
}

print("Rows and columns:", data.shape)
print("Date range:", data.index.min().date(), "to", data.index.max().date())
data.head()

Before modelling, check the structure. Missing values are not automatically zero. The assertions stop visibly if a condition needed by the walkthrough is false.


In [ ]:
print(data.dtypes)
print("\nMissing values:")
print(data.isna().sum())

assert data.index.is_monotonic_increasing
assert data.index.is_unique
assert not data.isna().any().any()
assert data["stress_index"].between(0, 1).all()
assert (data[assets] > -1).all().all()

print("\nAll structural checks passed.")

The upper panel below shows how €100 would have developed if it had been invested in one asset and left there. `(1 + return).cumprod()` successively multiplies the monthly growth factors. The lower panel shows the stress indicator. These plots describe the simulated history; they do not tell us what happens next.


In [ ]:
asset_growth = 100 * (1 + data[assets]).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]},
                         layout="constrained")

asset_growth.rename(columns=asset_labels).plot(ax=axes[0])
axes[0].set_ylabel("Value of €100")
axes[0].set_title("Synthetic asset histories")
axes[0].grid(alpha=0.2)

data["stress_index"].plot(ax=axes[1], color="black")
axes[1].set_ylabel("Stress")
axes[1].set_xlabel("Month")
axes[1].grid(alpha=0.2)
plt.show()

### A small return example

If an asset rises from €100 to €103, its return is `(103 - 100) / 100 = 0.03`, or 3%. `pct_change()` performs this calculation between consecutive rows. The first return is missing because no earlier price is available.


In [ ]:
price_example = pd.Series([100, 103, 101],
                          index=["Month 0", "Month 1", "Month 2"],
                          name="price")
pd.DataFrame({
    "price": price_example,
    "return": price_example.pct_change(),
})

**Self-check:** Why is the return from €103 to €101 only approximately −2%, not exactly −2%?  
It is approximately −1.94%, because the change of −€2 is divided by the starting value of €103, not by €100.


<h2 id="section-3-predict-next-months-returns">3. Predict next month's returns</h2>

Our observation is one decision month. The features known at its end are the four current returns and the current stress index. The targets are the four returns in the following month.

`shift(-1)` moves the next row's returns onto the current row. The final row then has no target because the dataset contains no later month, so we leave it out. This is a **multi-output regression**: one row of inputs produces four numerical predictions.


In [ ]:
feature_columns = assets + ["stress_index"]

X = data[feature_columns].iloc[:-1].copy()
y = data[assets].shift(-1).iloc[:-1].copy()

timeline_example = pd.concat(
    [X.add_prefix("known_"), y.add_prefix("next_")], axis=1
).head(4)
timeline_example

A random split would let early and late months mix. Instead, the final 24 targets form a future test period. All model and hyperparameter choices use the earlier development period.

Notice the timing: the last development target is observed at the first test decision date. At that month end it is legitimate to use that newly observed return to train the model before forecasting the following month.


In [ ]:
test_months = 24

X_development = X.iloc[:-test_months]
X_test = X.iloc[-test_months:]
y_development = y.iloc[:-test_months]
y_test = y.iloc[-test_months:]

split_summary = pd.DataFrame({
    "part": ["Development", "Future test"],
    "decision months": [len(X_development), len(X_test)],
    "first decision": [X_development.index.min(), X_test.index.min()],
    "last decision": [X_development.index.max(), X_test.index.max()],
})
split_summary

### Why Ridge regression?

Ordinary linear regression chooses coefficients that make the squared training errors small. Financial inputs can be correlated and future returns are noisy. The fitted coefficients can therefore react strongly to details of the particular training months.

**Ridge regression** adds a penalty for large coefficients:

$$
\text{loss} = \text{squared prediction errors} + \alpha \sum_j \beta_j^2
$$

The $\beta_j$ values are learned coefficients. `alpha` controls the penalty:

- `alpha = 0` is essentially ordinary linear regression;
- a larger `alpha` pulls coefficients more strongly towards zero;
- Ridge usually does not make coefficients exactly zero;
- moderate shrinkage can make predictions more stable on new data;
- excessive shrinkage can remove useful patterns.

This is a bias–variance trade-off: we accept a less flexible fit in exchange for possible stability. Scaling matters because the penalty acts on coefficient sizes. `StandardScaler` puts the input columns on comparable scales before Ridge is fitted.

**Self-check:** What should happen if `alpha` becomes extremely large?  
The coefficients become very small. Predictions then react little to the features and move towards a nearly constant prediction.


### Time-ordered validation

`TimeSeriesSplit` makes several expanding training periods. Every validation block occurs after its corresponding training block.

scikit-learn reports negative MAE because its search tools maximize scores. We put a minus sign in front when displaying it so that the familiar rule applies: **smaller MAE is better**.


In [ ]:
time_cv = TimeSeriesSplit(n_splits=4)

baseline = DummyRegressor(strategy="mean")
baseline_fold_mae = -cross_val_score(
    baseline,
    X_development,
    y_development,
    cv=time_cv,
    scoring="neg_mean_absolute_error",
)

ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge()),
])

alpha_values = np.geomspace(1e-3,1e3,100)
ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid={"model__alpha": alpha_values},
    cv=time_cv,
    scoring="neg_mean_absolute_error",
    refit=True,
)
ridge_search.fit(X_development, y_development)

cv_comparison = pd.DataFrame({
    "candidate": ["Mean baseline"] + [f"Ridge alpha={a}" for a in alpha_values],
    "mean_validation_MAE": [baseline_fold_mae.mean()]
        + list(-ridge_search.cv_results_["mean_test_score"]),
}).sort_values("mean_validation_MAE")

print("Selected Ridge setting:", ridge_search.best_params_)
cv_comparison

The following table isolates the shrinkage effect. Each candidate is fitted to the same complete development period after scaling. The Euclidean coefficient size summarizes all coefficients with one number. It is not an evaluation metric; it only shows how strongly Ridge shrinks the fitted relationship.


In [ ]:
coefficient_rows = []

for alpha in alpha_values:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha)),
    ])
    candidate.fit(X_development, y_development)
    coefficients = candidate.named_steps["model"].coef_
    coefficient_rows.append({
        "alpha": alpha,
        "coefficient_size": np.linalg.norm(coefficients),
    })

coefficient_comparison = pd.DataFrame(coefficient_rows)
coefficient_comparison

To see the effect of `alpha` on the size of the coefficients and the resulting MAE, we plot the two quantities on a log scale. The left axis shows the coefficient size and the right axis shows the MAE. The vertical line marks the best `alpha` found by the above cross-validation.

In [ ]:
ax1 = plt.gca()
ax2 = ax1.twinx()

ax1.plot(alpha_values, coefficient_comparison["coefficient_size"], color='blue')
ax1.set_xlabel('Ridge alpha')
ax1.set_ylabel('Coefficient size (L2 norm)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2.plot(alpha_values, -ridge_search.cv_results_["mean_test_score"], color='orange')
ax2.axhline(y=baseline_fold_mae.mean(), color='k', linestyle='--', label='Baseline MAE')
ax2.axvline(x=ridge_search.best_params_["model__alpha"], color='orange', linestyle='--', label='Best Ridge MAE')
ax2.set_ylabel('Mean Validation MAE', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

plt.xscale('log')
plt.xlim(alpha_values.min(), alpha_values.max())
plt.legend()

### Evaluate forecasts on future months

The search has already refitted the selected Ridge pipeline on all development observations. We fit the mean baseline to the same observations, then evaluate both once on the future test targets.


In [ ]:
baseline.fit(X_development, y_development)

baseline_test_predictions = baseline.predict(X_test)
ridge_test_predictions = ridge_search.predict(X_test)

prediction_results = pd.DataFrame({
    "model": ["Mean baseline", "Selected Ridge"],
    "future_test_MAE": [
        mean_absolute_error(y_test, baseline_test_predictions),
        mean_absolute_error(y_test, ridge_test_predictions),
    ],
})
prediction_results

MAE is expressed as a decimal monthly return. For example, `0.020` corresponds to an absolute error of about two percentage points. A modest improvement is plausible; perfect return prediction is not.

A better forecast MAE still does not guarantee a better portfolio. MAE averages errors across assets and months, while the optimizer reacts to the relative forecasts in each month.


<h2 id="section-4-from-forecasts-to-a-portfolio">4. From forecasts to a portfolio</h2>

Let $w$ contain the four portfolio weights and $\hat{\mu}$ the four predicted returns. The predicted portfolio return is:

$$
w^\top \hat{\mu}
$$

The `@` operator calculates this dot product. A 25% weight is stored as `0.25`. The weights must sum to 1.


In [ ]:
equal_weights = np.full(len(assets), 1 / len(assets))
first_forecast = ridge_test_predictions[0]

def portfolio_return(weights, asset_returns):
    return weights @ asset_returns

print("Equal weights:", equal_weights)
print("Sum of weights:", equal_weights.sum())
print("First Ridge forecast:", first_forecast.round(4))
print("Predicted equal-weight return:",
      round(portfolio_return(equal_weights, first_forecast), 4))

Risk depends not only on each asset's variation but also on how assets move together. The covariance matrix summarizes these relationships. We estimate it once from returns known at the start of the test exercise.

Portfolio variance is $w^\top \Sigma w$, where $\Sigma$ is the covariance matrix. Volatility is the square root of variance and therefore uses the same return units as the monthly data.


In [ ]:
first_test_decision = X_test.index[0]
known_returns = data.loc[:first_test_decision, assets]
covariance_matrix = known_returns.cov().to_numpy()
correlation_matrix = known_returns.corr().to_numpy()

def portfolio_variance(weights, covariance):
    return weights @ covariance @ weights

def portfolio_volatility(weights, covariance):
    return np.sqrt(portfolio_variance(weights, covariance))

covariance_table = pd.DataFrame(
    covariance_matrix,
    index=[asset_labels[a] for a in assets],
    columns=[asset_labels[a] for a in assets],
)

correlation_table = pd.DataFrame(
    correlation_matrix,
    index=[asset_labels[a] for a in assets],
    columns=[asset_labels[a] for a in assets],
)

print("Estimated covariance matrix:")
display(covariance_table.round(6))

print("Estimated correlation matrix:")
display(correlation_table.round(3))

print("Estimated monthly equal-weight volatility:",
      round(portfolio_volatility(equal_weights, covariance_matrix), 4))


**Self-check:** If two volatile assets always moved in exactly opposite directions, could combining them reduce portfolio volatility?  
Yes. Their negative covariance means losses in one tend to coincide with gains in the other. Real relationships are not perfect and market parameters can change, so estimated diversification is still uncertain!


<h2 id="section-5-optimize-one-portfolio">5. Optimize one portfolio</h2>

We ask SciPy to maximize predicted return while keeping estimated volatility no higher than the equal-weight portfolio. Because `minimize` minimizes a function, the objective returns the **negative** predicted return.

The decision rules are:

- weights sum to 1;
- every weight lies between 0 and 0.50;
- portfolio volatility is at most the equal-weight volatility.

An equality constraint returns zero when it is satisfied. An inequality constraint is satisfied when it returns a value greater than or equal to zero. Equal weights provide a feasible starting point.


In [ ]:
risk_limit = portfolio_volatility(equal_weights, covariance_matrix)
weight_bounds = [(0.0, 0.50)] * len(assets)

def optimize_portfolio(predicted_returns, covariance, maximum_volatility):
    def objective(weights):
        return -portfolio_return(weights, predicted_returns)

    # 1. The sum of weights must equal 1 (full investment).
    # 2. The portfolio volatility must not exceed the maximum allowed.
    constraints = [
        {"type": "eq", "fun": lambda weights: weights.sum() - 1},
        {
            "type": "ineq",
            "fun": lambda weights:
                maximum_volatility - portfolio_volatility(weights, covariance),
        },
    ]

    result = minimize(
        objective,
        x0=equal_weights,
        method="SLSQP",
        bounds=weight_bounds,
        constraints=constraints,
        options={"ftol": 1e-10, "maxiter": 1000},
    )

    if not result.success:
        raise RuntimeError(result.message)
    return result

first_result = optimize_portfolio(
    first_forecast, covariance_matrix, risk_limit
)
first_ml_weights = first_result.x
first_ml_volatility = portfolio_volatility(
    first_ml_weights, covariance_matrix
)

# Optimizers work with numerical tolerances rather than exact decimal equality.
# Check that the results are within reasonable bounds, allowing for small numerical errors.
assert np.isclose(first_ml_weights.sum(), 1.0)
assert (first_ml_weights >= -1e-8).all()
assert (first_ml_weights <= 0.50 + 1e-8).all()
assert first_ml_volatility <= risk_limit + 1e-8

first_portfolio = pd.DataFrame({
    "asset": [asset_labels[a] for a in assets],
    "equal_weight": equal_weights,
    "ML_weight": first_ml_weights,
    "predicted_return": first_forecast,
})

print("Optimization succeeded:", first_result.success)
print(f"Weight sum: {first_ml_weights.sum():.6f}")
print(f"Estimated volatility: {first_ml_volatility:.6f}")
print(f"Risk limit: {risk_limit:.6f}")
first_portfolio.round(4)

The optimizer may place more weight on an asset with an attractive forecast, but the volatility constraint and 50% cap restrict concentration. The result is thus conditional on both the predictions and the estimated covariance matrix.


In [ ]:
plot_weights = first_portfolio.set_index("asset")[["equal_weight", "ML_weight"]]
ax = plot_weights.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("Portfolio weight")
ax.set_title("One future decision: equal weights and ML-optimized weights")
ax.set_ylim(0, 0.55)
ax.grid(axis="y", alpha=0.2)
plt.show()

<h2 id="section-6-compare-strategies-on-future-months">6. Compare strategies on future months</h2>

We now repeat the same optimization for every test decision month. This is deliberately a **simplified** comparison:

- the Ridge model is fitted once before the test targets;
- the covariance matrix and risk limit stay fixed;
- current-month features become available sequentially;
- no transaction costs, taxes or trading delays are included.

The historical optimizer uses the same objective and constraints but replaces ML forecasts with the average returns known at the first test decision. Its weights therefore remain fixed. This isolates the effect of changing the return forecasts.


In [ ]:
historical_forecast = known_returns.mean().to_numpy()
historical_result = optimize_portfolio(
    historical_forecast, covariance_matrix, risk_limit
)
historical_weights = historical_result.x

ml_weights = []
for monthly_forecast in ridge_test_predictions:
    result = optimize_portfolio(
        monthly_forecast, covariance_matrix, risk_limit
    )
    ml_weights.append(result.x)
ml_weights = np.vstack(ml_weights)

realised_asset_returns = y_test.to_numpy()
strategy_returns = pd.DataFrame({
    "Equal weight": realised_asset_returns @ equal_weights,
    "Historical optimizer": realised_asset_returns @ historical_weights,
    "ML optimizer": np.sum(realised_asset_returns * ml_weights, axis=1),
})

# The target for a decision month is realised one month later.
decision_positions = data.index.get_indexer(X_test.index)
strategy_returns.index = data.index[decision_positions + 1]
strategy_returns.head()

We report the average and volatility of monthly returns plus the final value of an initial €100. These are descriptive results for 24 simulated future months, not guarantees. No annualisation is needed for this comparison.


In [ ]:
strategy_growth = 100 * (1 + strategy_returns).cumprod()

strategy_summary = pd.DataFrame({
    "average_monthly_return": strategy_returns.mean(),
    "monthly_volatility": strategy_returns.std(),
    "final_value_of_100": strategy_growth.iloc[-1],
})
strategy_summary.round(4)

In [ ]:
ax = strategy_growth.plot(figsize=(9, 5))
ax.set_ylabel("Portfolio value from €100")
ax.set_xlabel("Realisation month")
ax.set_title("Simplified out-of-sample strategy comparison")
ax.grid(alpha=0.2)
plt.show()

**Self-check:** If Ridge has the lowest prediction MAE, must the ML portfolio finish with the highest value?  
No. MAE averages forecast errors, whereas portfolio outcomes depend on which relative forecasts receive weight, the constraints, covariances and the particular realised returns. Twenty-four months also provide limited evidence.


<h2 id="section-7-sensitivity-and-limitations">7. Sensitivity and limitations</h2>

Optimizers can amplify small differences in estimated returns. We add 0.002, only 0.2 percentage points, to the first predicted global-equity return and optimize again.


In [ ]:
perturbed_forecast = first_forecast.copy()
perturbed_forecast[0] += 0.002

perturbed_result = optimize_portfolio(
    perturbed_forecast, covariance_matrix, risk_limit
)

sensitivity = pd.DataFrame({
    "asset": [asset_labels[a] for a in assets],
    "original_weight": first_ml_weights,
    "weight_after_small_forecast_change": perturbed_result.x,
    "change": perturbed_result.x - first_ml_weights,
})
sensitivity.round(4)

A small or large weight change is informative in either case. It describes sensitivity under these particular constraints; another month may react differently.

### What the example leaves out

- The data and stress indicator are simulated.
- Expected returns and covariances are estimated with uncertainty.
- Relationships can change over time.
- Variance treats positive and negative deviations symmetrically and is not every investor's complete definition of risk.
- The model ignores transaction costs, taxes and turnover.
- The fixed covariance matrix simplifies the test exercise.
- The 50% cap and equal-weight risk budget are choices, not laws of finance.
- A successful backtest does not demonstrate future profitability or causality.

Precise numerical weights are conditional recommendations from this simplified system. They should not be mistaken for precise knowledge of future returns.


<h2 id="section-8-recap-and-outlook">8. Recap and outlook</h2>

Answer in your own words:

1. What information does Ridge use, and what does it predict?
2. How does increasing `alpha` affect the fitted coefficients?
3. Why is the final test period later than the development period?
4. What is the difference between the mean prediction baseline and the equal-weight portfolio baseline?
5. What do the weight bounds and volatility constraint accomplish?
6. Why can a better MAE fail to produce a better portfolio?
7. What can and cannot be concluded from this synthetic test?

The complete workflow is:

**past information → return forecasts → constrained optimization → future evaluation**

Next week, dimensionality reduction will examine how information from several variables can be represented with fewer derived dimensions.


<h2 id="section-optional-your-turn">Optional: Your turn</h2>

Choose one investigation. State your expectation before changing the code, then explain the result and its limitations.

1. Change the maximum asset weight from 50% to 35%. Which portfolios change most?
2. Allow 10% more volatility than the equal-weight portfolio. Does the optimizer concentrate more strongly?
3. Increase a different forecast by 0.2 percentage points and compare the sensitivity.
4. Remove one asset and rerun the one-month optimization. How does the risk constraint affect feasibility?

These are explorations **after** the test results have been inspected. They should not be presented as a fresh, unbiased final evaluation.


In [ ]:
# Optional exploration workspace
# Copy the relevant objects before changing them so the walkthrough remains reproducible.


### Before you finish

1. Restart the kernel and run all cells from top to bottom.
2. Revisit any self-check whose answer was surprising.
3. Save and download your notebook before leaving Binder.
